In [28]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# TODO: Use single or double precision?
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path

import numpy as onp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from ase.visualize import view
from ase.atoms import Atoms

from msmjax.core.shortrange import make_eval_pair_pot, _gen_supercell
from msmjax.benchmark_tools import (
    evaluate_structure_with_lammps_p3m,
    path_input_structures,
    plot_parity_line,
)

LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

# Function definitions

In [67]:
def calc_energy_ref_nonperiodic(positions, charges):
    n_dim = positions.shape[1]
    compute_pair_term = make_eval_pair_pot(
        kernel_fn=lambda x: 1.0 / x, pbc=(False,) * n_dim
    )
    return compute_pair_term(positions, charges)


def calc_forces_ref_nonperiodic(positions, charges):
    return -jax.grad(calc_energy_ref_nonperiodic, argnums=0)(
        positions, charges
    )


@jax.jit
def calc_reference_results_nonperiodic(positions, charges):
    value, grad = jax.value_and_grad(calc_energy_ref_nonperiodic, argnums=0)(
        positions, charges
    )
    return value, -grad


# TODO: Use a single function that calculates both? (like for the periodic case)
calc_energy_ref_nonperiodic = jax.jit(calc_energy_ref_nonperiodic)
calc_forces_ref_nonperiodic = jax.jit(calc_forces_ref_nonperiodic)

In [62]:
def load_one_structure(n_particles):
    structures = onp.load(
        path_input_structures / ("structures_" + str(n_particles) + ".npz")
    )
    # TODO: Also test different structures of the same number of particles?
    #  (i.e., other values for idx_structure than 0)
    idx_structure = 0
    pos = structures["positions"][idx_structure]
    chg = structures["charges"][idx_structure]
    cell = structures["cells"][idx_structure]
    # TODO: Use single or double precision?
    pos = pos.astype(onp.float64)
    chg = chg.astype(onp.float64)
    cell = cell.astype(onp.float64)
    return pos, chg, cell

# Non-periodic

## Cubic

In [70]:
(pos, chg, cell) = load_one_structure(10000)

e_ref, f_ref = calc_reference_results_nonperiodic(pos, chg)

In [71]:
f_ref

Array([[ 0.73842372, -0.24257069, -0.35061577],
       [-1.2401413 , -0.74123898, -0.23865215],
       [-1.12433904, -2.55193567, -0.36900414],
       ...,
       [-0.31594839, -0.7829124 , -0.23782152],
       [ 0.92109785,  1.79489741, -0.07707081],
       [ 1.2867058 , -0.65347867, -1.75372884]], dtype=float64)

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the minimum number of grid points is reached along some axis before the others.

In [73]:
(pos, chg, cell) = load_one_structure(1500)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

In [74]:
atoms = Atoms(positions=pos, cell=cell)
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

In [75]:
e_ref, f_ref = calc_reference_results_nonperiodic(pos, chg)

f_ref

Array([[ 0.52291598, -0.50706152, -0.13971596],
       [ 1.26399352, -0.11866261,  1.59667003],
       [-0.49605666, -1.04739648,  1.94566818],
       ...,
       [ 0.38414768,  0.32880002,  0.36100079],
       [ 0.16111263, -0.25890213,  0.03127283],
       [ 0.61888895,  0.07044311,  0.72569831]], dtype=float64)

## Triclinic

In [76]:
(pos, chg, cell) = load_one_structure(10000)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.8, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

In [77]:
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

In [78]:
e_ref, f_ref = calc_reference_results_nonperiodic(pos, chg)

f_ref

Array([[ 0.72090493,  0.02703103, -0.03089184],
       [-1.00961552, -1.09905832,  0.0543119 ],
       [-0.54966596, -3.59848403,  0.74814954],
       ...,
       [-0.54304843, -0.44507136, -0.25847148],
       [ 0.5069535 ,  1.05381282, -1.86078862],
       [ 0.89049978, -0.61461811, -0.75983057]], dtype=float64)

# Periodic

## Cubic

In [41]:
(pos, chg, cell) = load_one_structure(500)

In [52]:
e_ref, f_ref = evaluate_structure_with_lammps_p3m(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

In [53]:
f_ref

array([[ 0.14806545,  0.1164612 , -0.0651557 ],
       [ 2.21022108, -0.30590684, -1.03799778],
       [ 0.65170702, -2.3006747 , -1.16444533],
       ...,
       [-0.48133338, -0.06234535, -1.97686117],
       [ 0.04829195, -0.23919825,  0.15981921],
       [-0.59151248, -0.40433773, -1.06342896]])

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the grid is reduced to a single point along some axis faster than along the others.

In [56]:
(pos, chg, cell) = load_one_structure(500)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

In [59]:
atoms = Atoms(positions=pos, cell=cell)
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

In [60]:
e_ref, f_ref = evaluate_structure_with_lammps_p3m(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

In [61]:
f_ref

array([[ 0.03977737,  0.05076368, -0.13066016],
       [ 2.08446104, -0.34492516, -1.01916398],
       [ 0.76243544, -2.44417129, -0.98002413],
       ...,
       [-0.73138606,  0.07339625, -1.93341566],
       [-0.03455696, -0.25096243,  0.20930446],
       [-0.42340001, -0.52829078, -0.75976871]])

## Triclinic

In [63]:
(pos, chg, cell) = load_one_structure(500)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.8, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

In [64]:
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

In [65]:
e_ref, f_ref = evaluate_structure_with_lammps_p3m(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)